In [1]:
import collections
import dataclasses
import logging
import math
import pathlib

import imageio
from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
import numpy as np
import tqdm
import tyro

from myutils.pi0_infer import Pi0Inference

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
2025-06-25 09:35:04.870846: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-25 09:35:04.887478: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750836904.906073  617803 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been reg

In [2]:
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
LIBERO_ENV_RESOLUTION = 256  # resolution used to render training data

In [3]:

@dataclasses.dataclass
class Args:
    #################################################################################################################
    # Model server parameters
    #################################################################################################################
    pretrained_model_path: str = "outputs/train/2025-06-24/16-28-13_libero_noops_finetune_sameconfig/checkpoints/last/pretrained_model"
    resize_size: int = 256
    replan_steps: int = 16

    #################################################################################################################
    # LIBERO environment-specific parameters
    #################################################################################################################
    task_suite_name: str = (
        "libero_10"  # Task suite. Options: libero_spatial, libero_object, libero_goal, libero_10, libero_90
    )
    num_steps_wait: int = 10  # Number of steps to wait for objects to stabilize i n sim
    num_trials_per_task: int = 1  # Number of rollouts per task

    #################################################################################################################
    # Utils
    #################################################################################################################
    video_out_path: str = "data/libero/videos"  # Path to save videos

    seed: int = 7  # Random Seed (for reproducibility)


In [4]:

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description


def _quat2axisangle(quat):
    """
    Copied from robosuite: https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py#L490C1-L512C55
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den


In [5]:
args = Args()

In [6]:
# Set random seed
np.random.seed(args.seed)

# Initialize LIBERO task suite
benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[args.task_suite_name]()
num_tasks_in_suite = task_suite.n_tasks
logging.info(f"Task suite: {args.task_suite_name}")

pathlib.Path(args.video_out_path).mkdir(parents=True, exist_ok=True)

if args.task_suite_name == "libero_spatial":
    max_steps = 220  # longest training demo has 193 steps
elif args.task_suite_name == "libero_object":
    max_steps = 280  # longest training demo has 254 steps
elif args.task_suite_name == "libero_goal":
    max_steps = 300  # longest training demo has 270 steps
elif args.task_suite_name == "libero_10":
    max_steps = 520  # longest training demo has 505 steps
elif args.task_suite_name == "libero_90":
    max_steps = 400  # longest training demo has 373 steps
else:
    raise ValueError(f"Unknown task suite: {args.task_suite_name}")

mypolicy = Pi0Inference(args.pretrained_model_path)



[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Loading weights from local directory


In [7]:
def normalize_gripper_action(action, binarize=True):
    """
    Changes gripper action (last dimension of action vector) from [0,1] to [-1,+1].
    Necessary for some environments (not Bridge) because the dataset wrapper standardizes gripper actions to [0,1].
    Note that unlike the other action dimensions, the gripper action is not normalized to [-1,+1] by default by
    the dataset wrapper.

    Normalization formula: y = 2 * (x - orig_low) / (orig_high - orig_low) - 1
    """
    # Just normalize the last action to [-1,+1].
    orig_low, orig_high = 0.0, 1.0
    action[..., -1] = 2 * (action[..., -1] - orig_low) / (orig_high - orig_low) - 1

    if binarize:
        # Binarize to -1 or +1.
        action[..., -1] = np.sign(action[..., -1])

    return action

def invert_gripper_action(action):
    """
    Flips the sign of the gripper action (last dimension of action vector).
    This is necessary for some environments where -1 = open, +1 = close, since
    the RLDS dataloader aligns gripper actions such that 0 = close, 1 = open.
    """
    action[..., -1] = action[..., -1] * -1.0
    return action

# Start evaluation
total_episodes, total_successes = 0, 0
for task_id in tqdm.tqdm(range(num_tasks_in_suite)):
    # Get task
    task = task_suite.get_task(task_id)

    # Get default LIBERO initial states
    initial_states = task_suite.get_task_init_states(task_id)

    # Initialize LIBERO environment and task description
    env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, args.seed)

    # Start episodes
    task_episodes, task_successes = 0, 0
    for episode_idx in tqdm.tqdm(range(args.num_trials_per_task)):
        logging.info(f"\nTask: {task_description}")

        # Reset environment
        env.reset()
        action_plan = collections.deque()

        # Set initial states
        obs = env.set_init_state(initial_states[episode_idx])

        # Setup
        t = 0
        replay_images = []

        logging.info(f"Starting episode {task_episodes+1}...")
        while t < max_steps + args.num_steps_wait:
            try:
                # IMPORTANT: Do nothing for the first few timesteps because the simulator drops objects
                # and we need to wait for them to fall
                if t < args.num_steps_wait:
                    obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)
                    t += 1
                    continue

                # Get preprocessed image
                # IMPORTANT: rotate 180 degrees to match train preprocessing
                img = np.ascontiguousarray(obs["agentview_image"][::-1, ::-1])
                wrist_img = np.ascontiguousarray(obs["robot0_eye_in_hand_image"][::-1, ::-1])
                # from PIL import Image
                # Image.fromarray(img).save("img.jpg")
                # Image.fromarray(wrist_img).save("wrist_img.jpg")

                # img = image_tools.convert_to_uint8(
                #     image_tools.resize_with_pad(img, args.resize_size, args.resize_size)
                # )
                # wrist_img = image_tools.convert_to_uint8(
                #     image_tools.resize_with_pad(wrist_img, args.resize_size, args.resize_size)
                # )

                # Save preprocessed image for replay video
                replay_images.append(img)

                if not action_plan:
                    # Finished executing previous action chunk -- compute new chunk
                    # Prepare observations dict
                    element = {
                        "observation.images.image": img,
                        "observation.images.wrist_image": wrist_img,
                        "observation.state": np.concatenate(
                            (
                                obs["robot0_eef_pos"],
                                _quat2axisangle(obs["robot0_eef_quat"]),
                                obs["robot0_gripper_qpos"],
                            )
                        ),
                        "task": str(task_description),
                    }

                    # Query model to get action
                    action_chunk = mypolicy.infer(element)
                    assert (
                        len(action_chunk) >= args.replan_steps
                    ), f"We want to replan every {args.replan_steps} steps, but policy only predicts {len(action_chunk)} steps."
                    action_plan.extend(action_chunk[: args.replan_steps])

                action = action_plan.popleft()
                action = normalize_gripper_action(action, binarize=True)
                action = invert_gripper_action(action) # for libero, -1=open, 1=close

                # Execute action in environment
                obs, reward, done, info = env.step(action.tolist())
                if done:
                    task_successes += 1
                    total_successes += 1
                    break
                t += 1

            except Exception as e:
                logging.error(f"Caught exception: {e}")
                break

        task_episodes += 1
        total_episodes += 1

        # Save a replay video of the episode
        suffix = "success" if done else "failure"
        task_segment = task_description.replace(" ", "_")
        imageio.mimwrite(
            pathlib.Path(args.video_out_path) / f"rollout_{task_segment}_{suffix}.mp4",
            [np.asarray(x) for x in replay_images],
            fps=10,
        )

        # Log current results
        logging.info(f"Success: {done}")
        logging.info(f"# episodes completed so far: {total_episodes}")
        logging.info(f"# successes: {total_successes} ({total_successes / total_episodes * 100:.1f}%)")

    # Log final results
    logging.info(f"Current task success rate: {float(task_successes) / float(task_episodes)}")
    logging.info(f"Current total success rate: {float(total_successes) / float(total_episodes)}")

logging.info(f"Total success rate: {float(total_successes) / float(total_episodes)}")
logging.info(f"Total episodes: {total_episodes}")


  0%|          | 0/10 [00:00<?, ?it/s]

[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!
[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 10%|█         | 1/10 [00:28<04:13, 28.12s/it]

[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!
[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 20%|██        | 2/10 [00:51<03:23, 25.48s/it]

...

[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!
[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 70%|███████   | 7/10 [03:08<01:31, 30.39s/it]

[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!
[Warning]: datasets path /mnt/lustre-grete/usr/u12045/vla/duci/pi0_lerobo/libero/libero/datasets does not exist!


 70%|███████   | 7/10 [03:13<01:22, 27.65s/it]


KeyboardInterrupt: 